In [27]:
import sys
from pathlib import Path

# Repo root auf sys.path — funktioniert ob Kernel-cwd notebooks/ oder Root ist
_root = Path().resolve()
if not (_root / "qc").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
from qc.instance import load_instance
from qc.grover_mixer import enumerate_feasible
from qc.dense import grover_mixer_subspace

_DATA = str(_root / "artifacts/data/all_data.csv")
inst = load_instance(_DATA, start=645, T=2)
feas = enumerate_feasible(inst)
U = grover_mixer_subspace(beta=np.pi/2, dim=len(feas))

In [28]:
import plotly.express as px

# U ist komplex, Imaginärteil überall konstant -> Realteil zeigt die Struktur
vmax = np.abs(U.real).max()
fig = px.imshow(
    U.real,
    color_continuous_scale="RdBu_r",
    zmin=-vmax,
    zmax=vmax,
    aspect="equal",
    labels={"color": "Re(U)", "x": "Spalte j", "y": "Zeile i"},
    title="Grover-Mixer U (Realteil)",
)
fig.update_layout(width=650, height=600)
fig.show()

In [29]:
U.shape

(486, 486)

In [30]:
# Ecke ansehen
np.round(U[:3, :3], 5)

array([[ 0.99794-0.00206j, -0.00206-0.00206j, -0.00206-0.00206j],
       [-0.00206-0.00206j,  0.99794-0.00206j, -0.00206-0.00206j],
       [-0.00206-0.00206j, -0.00206-0.00206j,  0.99794-0.00206j]])

In [31]:
np.unique(np.round(U, 10)).size

2

# GM-QAOA: Aufbau und Ablauf

Die zwei Bausteine werden getrennt konstruiert und dann im QAOA-Schaltkreis abwechselnd angewandt:

```text
 Feasibility-Prädikat (qc/instance.py)            Kosten je Konfiguration z
 Lade/Entlade-XOR, Import/Export-XOR,             direkte Kosten (z-only; Runde 1 einziger
 SoC-One-Hot, Outage-Pinning                      Term: Resiliency-Erlös −225 $ je bedientem
          │                                       Outage-Slot) + max über Benders-Cuts (ab R2)
          │ Oracle über 2^8 Muster je Slot                 │
          ▼                                                ▼
 per-Slot-Mengen: 27 (online) / 18 (outage)        H_C = diag(E(z)),  z ∈ F
          │                                        prägt nur Phasen ein,
          │ kartesisches Produkt entlang g_avail   mischt nichts
          ▼
 F: 27 × 18 = 486 feasible Zustände
          │
          ▼
 |F⟩ = Gleichverteilung über F
 U_M(β) = I − (1 − e^{−iβ}) |F⟩⟨F|
 mischt all-to-all innerhalb F,
 koppelt nie nach außen
```

Der Schaltkreis alterniert beide Operatoren $p$-mal, mit festen **Ramp-Winkeln** (annealing-artig: $\gamma$ steigt, $\beta$ fällt — kein Parameter-Training):

$$|\psi\rangle \;=\; \underbrace{U_M(\beta_p)\, e^{-i\gamma_p H_C}}_{\text{Layer } p} \cdots \underbrace{U_M(\beta_1)\, e^{-i\gamma_1 H_C}}_{\text{Layer } 1}\; |F\rangle, \qquad P(z) = |\psi_z|^2$$

Rollenverteilung: $H_C$ **schreibt Kosteninformation in die Phasen**, der Mixer **übersetzt Phasendifferenzen in Betragsdifferenzen** (Spiegelung am Amplituden-Mittelwert). Keiner der beiden kann allein eine Verteilung formen — Phasen ändern keine Wahrscheinlichkeiten, und der Mixer auf phasengleichem Zustand ist nur eine globale Phase. Die drei Plots darunter zeigen die Bausteine einzeln: das Oracle (Mixer-Konstruktion), die Diagonalstruktur von $H_C$, und die Amplifikation über die Layer.

In [8]:
from qc.instance import BITS_PER_SLOT, Instance, int_to_bits, structurally_feasible

# 1) Mixer-Konstruktion: das Feasibility-Prädikat als Oracle über alle 2^8 = 256
#    Bitmuster EINES Slots — einmal für online (g=1), einmal für outage (g=0)
patterns = np.arange(2 ** BITS_PER_SLOT)
mask = {}
for g in (1, 0):
    dummy = Instance(p_pv=np.zeros(1), p_load=np.zeros(1),
                     tou=np.zeros(1), g_avail=np.array([g]))
    mask[g] = structurally_feasible(int_to_bits(patterns, BITS_PER_SLOT), dummy)

n_on, n_off = int(mask[1].sum()), int(mask[0].sum())
print(f"online: {n_on}/256 feasibel, outage: {n_off}/256 feasibel")
print(f"F = kartesisches Produkt entlang g_avail = {n_on} x {n_off} = {n_on * n_off} Zustände")

fig = px.imshow(
    np.vstack([mask[1], mask[0]]).astype(int),
    color_continuous_scale=[[0.0, "#e1e0d9"], [1.0, "#2a78d6"]],
    aspect="auto",
    labels={"x": "8-Bit-Slotmuster (0–255)", "y": ""},
    y=["online (g=1)", "outage (g=0)"],
    title="Oracle je Slot-Typ: welche der 256 Slotmuster sind strukturell feasibel?",
)
fig.update_layout(width=750, height=230, coloraxis_showscale=False)
fig.show()

online: 27/256 feasibel, outage: 18/256 feasibel
F = kartesisches Produkt entlang g_avail = 27 x 18 = 486 Zustände


In [9]:
from qc.instance import direct_costs

# 2) Cost-Hamiltonian: H_C ist DIAGONAL im Zustandsraum — Eintrag [z,z] = Kosten von z
#    (Runde 1: nur direkte Kosten; spätere Runden: + max über Benders-Cuts).
#    Er mischt nichts, er praegt nur Phasen ein: e^{-i gamma H_C} wirkt elementweise.
E_direct = direct_costs(int_to_bits(feas, inst.n_bits), inst)
H_C = np.diag(E_direct)

# Fenster um Index 243: dort springt die Diagonale von 0 (y=0) auf -225 (y=1, served)
lo, hi = 233, 253
win = list(range(lo, hi))
fig = px.imshow(
    H_C[lo:hi, lo:hi], x=win, y=win,
    color_continuous_scale=[[0.0, "#104281"], [1.0, "#f0efec"]],
    aspect="equal",
    labels={"color": "H_C[z,z] ($)", "x": "Spalte j", "y": "Zeile i"},
    title="Cost-Hamiltonian H_C (Ausschnitt): nur Diagonale besetzt — vgl. Mixer oben (voll besetzt)",
)
fig.update_layout(width=650, height=600)
fig.show()

In [10]:
import plotly.graph_objects as go

from qc.qaoa import normalize_costs, ramp_angles

# 3) Der QAOA-Ablauf: p Layer mit festen Ramp-Winkeln (kein Training),
#    gamma steigt (Kosten zählen mehr), beta fällt (Mixer wird schwächer)
p = 6
gammas, betas = ramp_angles(p, gamma_max=4 * np.pi, beta_max=2 * np.pi)

E = normalize_costs(E_direct)  # Kosten -> [0, 1]
psi = np.full(len(feas), 1 / np.sqrt(len(feas)), dtype=complex)  # Start: |F>
hist = [np.abs(psi) ** 2]
for g_, b_ in zip(gammas, betas):
    psi = np.exp(-1j * g_ * E) * psi                      # Kostenphase e^{-i gamma H_C}
    psi = psi - (1 - np.exp(-1j * b_)) * psi.mean()       # Grover-Mixer U_M(beta)
    hist.append(np.abs(psi) ** 2)

P_min = [float(h[E == 0].sum()) for h in hist]   # Wahrscheinlichkeit der Minimalkosten-Gruppe
P_rest = [1 - v for v in P_min]

layers = list(range(p + 1))
fig = go.Figure()
fig.add_hline(y=0.5, line_dash="dash", line_color="#898781",
              annotation_text="uniform (Start)", annotation_font_color="#52514e")
fig.add_scatter(x=layers, y=P_min, mode="lines+markers",
                line=dict(color="#2a78d6", width=2), marker=dict(size=8),
                name="P(Minimalkosten-Gruppe)")
fig.add_scatter(x=layers, y=P_rest, mode="lines+markers",
                line=dict(color="#1baf7a", width=2), marker=dict(size=8),
                name="P(Rest)")
fig.update_layout(
    title=f"QAOA-Ablauf: Amplifikation über {p} Layer (Interferenz, nicht monoton)",
    xaxis_title="nach Layer k", yaxis_title="Gruppenwahrscheinlichkeit",
    xaxis=dict(tickmode="linear"), width=750, height=420, plot_bgcolor="#fcfcfb",
)
fig.show()

## Vom Mixer + Cost-Hamiltonian zur Verteilung

Der Zustand $\psi$ lebt nur auf den 486 feasiblen Zuständen — der Mixer koppelt nie nach außen, infeasible Amplituden bleiben exakt 0. Start ist die Gleichverteilung $|F\rangle$: $\psi_z = 1/\sqrt{486}$ für alle $z$.

Ein QAOA-Layer ([qaoa.py:77-79](qc/qaoa.py#L77-L79)):

1. **Kostenphase** — $\psi_z \leftarrow e^{-i\gamma E(z)}\,\psi_z$, mit $H_C$ diagonal und $E(z)$ = auf $[0,1]$ normierte Kosten. Ändert nur die *Phase*, nie $|\psi_z|$.
2. **Grover-Mixer** — $\psi \leftarrow \psi - (1 - e^{-i\beta})\cdot\mathrm{mean}(\psi)$, d. h. $U_M = I - (1-e^{-i\beta})|F\rangle\langle F|$: eine Spiegelung am Mittelwert aller Amplituden.

Nach $p$ Layern wird gemessen: $P(z) = |\psi_z|^2$.

Der Mechanismus: Schritt 1 allein ändert keine einzige Wahrscheinlichkeit. Erst der Mixer macht aus **Phasendifferenzen Betragsdifferenzen** — Zustände, deren Phase nahe an $\mathrm{mean}(\psi)$ liegt, interferieren konstruktiv, die anderen destruktiv. Ohne Kostendifferenzen gibt es keine Phasendifferenzen, und der Mixer hat nichts zu verstärken.

In [11]:
from qc.instance import bit_index, direct_costs, int_to_bits
from qc.qaoa import gm_qaoa, normalize_costs

# Runde 1 des Benders-Loops: H_C-Diagonale = nur direkte (z-only) Kosten
bits = int_to_bits(feas, inst.n_bits)
costs1 = direct_costs(bits, inst)

t_out = int(np.where(inst.g_avail == 0)[0][0])        # Slot 646 ist der Outage-Slot
served = bits[:, bit_index(t_out, "y")].astype(bool)  # y-Bit im Outage-Slot

print("g_avail =", inst.g_avail.tolist(), "| Outage in Slot t =", t_out)
print("Kostenlevel:", np.unique(costs1), "-> je", int(served.sum()), "/", int((~served).sum()), "Zustände")

g_avail = [1, 0] | Outage in Slot t = 1
Kostenlevel: [-225.    0.] -> je 243 / 243 Zustände


In [12]:
# Ein QAOA-Layer von Hand (gamma = beta = pi/2), Start: Gleichverteilung auf |F>
E = normalize_costs(costs1)                               # {-225, 0} -> {0, 1}
psi = np.full(len(feas), 1 / np.sqrt(len(feas)), dtype=complex)

psi = np.exp(-1j * (np.pi / 2) * E) * psi                 # 1) Kostenphase: dreht nur die y=0-Gruppe (E=1)
psi = psi - (1 - np.exp(-1j * (np.pi / 2))) * psi.mean()  # 2) Grover-Mixer: Spiegelung am Mittelwert

np.unique(np.round(np.abs(psi) ** 2, 12))                 # genau zwei Werte: 0 und 1/243

array([0.        , 0.00411523])

### Warum bei einem Outage die Wahrscheinlichkeiten ungleich sind — und ohne Outage nicht

**Woher die Kostenlevel überhaupt kommen:** Die Runde-1-Diagonale enthält nur die *direkten* (z-only) Kosten, und davon gibt es genau einen Term — den **Resiliency-Erlös von −225 \$ pro bedientem Outage-Slot** (`y=1`, siehe `qc/instance.py::direct_costs`). Er ist der einzige Kostenanteil, der allein von einem Bit abhängt und nicht von den kontinuierlichen Leistungswerten. Alles andere (ToU-Energie, Demand Charge, Export-Erlös) hängt von `x` ab und kommt erst ab Runde 2 über Benders-Cuts in $H_C$. Ladeprofil, Import/Export-Bits und SoC-Band ändern das Kostenlevel in Runde 1 deshalb nicht.

**Ohne Outage:** Das y-Bit ist per Feasibility auf 0 gepinnt ⇒ der Resiliency-Term entfällt ⇒ alle direkten Kosten sind 0 ⇒ $E \equiv 0$ ⇒ die Kostenphase ist die Identität. Der Mixer angewandt auf den uniformen Zustand ergibt nur eine globale Phase ($\psi_z = \mathrm{mean}(\psi)$ für alle $z$). Es gibt nichts zu unterscheiden — die Verteilung bleibt exakt uniform, $P(z) = 1/486$. (Kein Bug: Runde 1 hat ohne Outage schlicht noch keine Kosteninformation; die kommt erst mit dem ersten Cut.)

**Mit Outage:** Das served-Bit $y$ im Outage-Slot bringt −225 \$ ⇒ zwei Kostenlevel $\{-225, 0\}$ ⇒ nach Normierung $E \in \{0, 1\}$. Die Kostenphase dreht nur die $y=0$-Gruppe; $\mathrm{mean}(\psi)$ liegt danach „zwischen“ beiden Gruppen. Die Spiegelung am Mittelwert interferiert für die günstige Gruppe konstruktiv, für die teure destruktiv — bei $\gamma = \beta = \pi/2$ und dem 50/50-Split hier sogar exakt: die $y=0$-Zustände landen auf Amplitude 0, die $y=1$-Zustände auf $1/\sqrt{243}$.

Entscheidend: $P(z)$ ist eine reine Funktion von $E(z)$ — **gleiche Kosten ⇒ exakt gleiche Wahrscheinlichkeit**. Deshalb gibt es genau so viele verschiedene $P$-Werte wie Kostenlevel: in Runde 1 zwei, entlang des einen y-Bits.

In [ ]:
import plotly.graph_objects as go

# Volle Evolution mit den kalibrierten Defaults (p=6, Ramp-Winkel aus qc/qaoa.py)
probs_r1 = gm_qaoa(costs1, p=6)

# Kontrast: ohne Outage sind alle direkten Kosten 0 -> Verteilung bleibt exakt uniform
probs_flat = gm_qaoa(np.zeros(len(feas)), p=6)
print("ohne Outage:", np.unique(np.round(probs_flat, 12)), "= 1/486 überall")
print("mit Outage: ", np.unique(np.round(probs_r1, 12)), "-> genau 2 Werte")
print("P(y=1-Menge) =", round(float(probs_r1[served].sum()), 4), "| uniform wäre 0.5")

idx = np.arange(len(feas))
fig = go.Figure()
fig.add_hline(y=1 / len(feas), line_dash="dash", line_color="#898781",
              annotation_text="uniform 1/|F|", annotation_font_color="#52514e")
fig.add_scatter(x=idx[served], y=probs_r1[served], mode="markers",
                marker=dict(color="#2a78d6", size=5), name="y=1 (served, -225 $)")
fig.add_scatter(x=idx[~served], y=probs_r1[~served], mode="markers",
                marker=dict(color="#1baf7a", size=5), name="y=0 (0 $)")
fig.update_layout(
    title="Runde 1 (GM-QAOA p=6): zwei Kostenlevel ⇒ zwei Wahrscheinlichkeitswerte",
    xaxis_title="Index des feasiblen Zustands", yaxis_title="P(z)",
    width=750, height=420, plot_bgcolor="#fcfcfb",
)
fig.show()

ohne Outage: [0.00205761] = 1/486 überall
mit Outage:  [4.62677110e-05 4.06895863e-03] -> genau 2 Werte
P(y=1-Menge) = 0.9888 | uniform wäre 0.5


### Runde 1: Ein Sample ziehen und dekodieren

Eine Messung des Quantenregisters kollabiert $\psi$ auf genau **einen** Basiszustand $z$, mit Wahrscheinlichkeit $P(z) = |\psi_z|^2$. Im Simulator ist das ein Zug aus der berechneten Verteilung (`rng.choice(..., p=probs)`). Wir messen `shots`-mal und nehmen das nach aktuellem Kostenmodell günstigste gesampelte $z$ — **best-of-shots**, wie in `qc/qaoa.py::sample_best`. In Runde 1 landet man mit $P \approx 0.989$ fast sicher in der served-Gruppe.

Das gezogene $z$ ist ein 16-Bit-Integer, **8 Bit pro Slot, LSB-first**: Bit $b = 8t + \text{Rollenoffset}$ (siehe `qc/instance.py::ROLES`). `decode(z, inst)` zerlegt es in die Rollenbelegung pro Slot — genau diese Belegung wird anschließend im Gurobi-Subproblem fixiert:

| Bit (Slot $t$) | Rolle | Bedeutung |
|---|---|---|
| $8t+0$ | `ch` | Batterie lädt |
| $8t+1$ | `dis` | Batterie entlädt (XOR mit `ch`: höchstens eins) |
| $8t+2$ | `imp` | Import aus dem Netz (Outage: auf 0 gepinnt) |
| $8t+3$ | `exp` | Export ins Netz (XOR mit `imp`; Outage: auf 0 gepinnt) |
| $8t+4..6$ | `b_low/b_mid/b_high` | SoC-Band, one-hot (genau eins = 1) |
| $8t+7$ | `y` | Last im Outage-Slot bedient → Resiliency-Erlös −225 \$ (online: auf 0 gepinnt) |

In [14]:
from qc.instance import ROLES, decode

ROLE_MEANING = {
    "ch":     "Batterie lädt",
    "dis":    "Batterie entlädt",
    "imp":    "Import aus dem Netz (bei Outage auf 0 gepinnt)",
    "exp":    "Export ins Netz (bei Outage auf 0 gepinnt)",
    "b_low":  "SoC-Band niedrig  ┐",
    "b_mid":  "SoC-Band mittel   ├ one-hot: genau eins = 1",
    "b_high": "SoC-Band hoch     ┘",
    "y":      "Last im Outage-Slot bedient (Resiliency-Erlös −225 $; online auf 0 gepinnt)",
}

# Messung simulieren: 1024 Shots aus der QAOA-Verteilung ziehen,
# dann best-of-shots nach aktuellem Kostenmodell (== qc/qaoa.py::sample_best)
rng = np.random.default_rng()
shots = 1024
shot_idx = rng.choice(len(feas), size=shots, p=probs_r1)
best_idx = shot_idx[np.argmin(costs1[shot_idx])]
z = int(feas[best_idx])

print(f"{shots} Shots -> {np.unique(shot_idx).size} verschiedene Zustände gesehen")
print(f"bestes Sample: z = {z}, Bitstring {z:016b} (links Bit 15, rechts Bit 0), "
      f"Kosten {costs1[best_idx]:.0f} $\n")

for t, slot in enumerate(decode(z, inst)):
    status = "online" if inst.g_avail[t] else "OUTAGE"
    print(f"Slot {t} ({status}) — Bits {8 * t}..{8 * t + 7}:")
    for role in ROLES:
        marker = "x" if slot[role] else " "
        print(f"  [{marker}] Bit {bit_index(t, role):>2}  {role:<7} = {slot[role]}   {ROLE_MEANING[role]}")
    print()

1024 Shots -> 252 verschiedene Zustände gesehen
bestes Sample: z = 37397, Bitstring 1001001000010101 (links Bit 15, rechts Bit 0), Kosten -225 $

Slot 0 (online) — Bits 0..7:
  [x] Bit  0  ch      = 1   Batterie lädt
  [ ] Bit  1  dis     = 0   Batterie entlädt
  [x] Bit  2  imp     = 1   Import aus dem Netz (bei Outage auf 0 gepinnt)
  [ ] Bit  3  exp     = 0   Export ins Netz (bei Outage auf 0 gepinnt)
  [x] Bit  4  b_low   = 1   SoC-Band niedrig  ┐
  [ ] Bit  5  b_mid   = 0   SoC-Band mittel   ├ one-hot: genau eins = 1
  [ ] Bit  6  b_high  = 0   SoC-Band hoch     ┘
  [ ] Bit  7  y       = 0   Last im Outage-Slot bedient (Resiliency-Erlös −225 $; online auf 0 gepinnt)

Slot 1 (OUTAGE) — Bits 8..15:
  [ ] Bit  8  ch      = 0   Batterie lädt
  [x] Bit  9  dis     = 1   Batterie entlädt
  [ ] Bit 10  imp     = 0   Import aus dem Netz (bei Outage auf 0 gepinnt)
  [ ] Bit 11  exp     = 0   Export ins Netz (bei Outage auf 0 gepinnt)
  [x] Bit 12  b_low   = 1   SoC-Band niedrig  ┐
  [ ] Bi

### Runde 2: Der Benders-Cut macht die Kosten kontinuierlich

Nach Runde 1 fixiert Gurobi das gesampelte `z`, löst das kontinuierliche LP und liefert Schattenpreise $\lambda$. Daraus entsteht der Optimality-Cut

$$q(z) \;\ge\; q_0 + \sum_b \lambda_b \, z_b$$

— affin in den Bits, aber mit **kontinuierlichen** Koeffizienten (Dual-Werte in \$). Die $H_C$-Diagonale der Runde 2 ist: direkte Kosten + punktweises Maximum aller bisherigen Cuts.

Damit hat (fast) jeder Zustand seinen **eigenen** Kostenwert $E(z)$ ⇒ eigene Phase $e^{-i\gamma E(z)}$ ⇒ eigenes Interferenzmuster gegen $\mathrm{mean}(\psi)$ ⇒ eigener $P(z)$-Wert. Die Zwei-Level-Struktur aus Runde 1 löst sich in ein kontinuierliches, mit den Kosten fallendes Spektrum auf — genau das, was der Benders-Loop braucht: das Sampling konzentriert sich zunehmend auf die nach aktuellem Kostenmodell günstigen Konfigurationen.

In [ ]:
# Runde 2: erster ECHTER Optimality-Cut aus dem Gurobi-Subproblem (Aufgabe 8).
# Der Cut ist affin in z: q(z) >= q̄ + w·(z − z̄) — w entsteht aus den
# LP-Schattenpreisen π und der affinen RHS-Map (z steckt nur in den RHS).
from subproblem.subproblem import solve_subproblem
from qc.benders import FEAS_TOL, build_sub_instance, feasibility_cut, optimality_cut

idx_z = int(np.where(feas == z)[0][0])
res = solve_subproblem(build_sub_instance(inst, z))

if not res.feasible:
    # Das gesampelte z hat KEINE kontinuierliche Fortsetzung — genau der
    # Feasibility-Cut-Fall: der Farkas-Ray eliminiert alle z mit demselben Beweis.
    fcut = feasibility_cut(res, bits[idx_z], inst.n_bits)
    keep = fcut.evaluate(bits) >= -FEAS_TOL if fcut is not None else feas != z
    print(f"z={z:0{inst.n_bits}b} infeasibel → Feasibility-Cut eliminiert {int((~keep).sum())} "
          f"von {len(feas)} States")
    # Für die Optimality-Cut-Illustration: günstigstes z mit feasiblem Subproblem.
    for cand in feas[np.argsort(costs1)]:
        res = solve_subproblem(build_sub_instance(inst, int(cand)))
        if res.feasible:
            z = int(cand)
            idx_z = int(np.where(feas == z)[0][0])
            break

cut = optimality_cut(res, bits[idx_z], inst.n_bits)
vals = cut.evaluate(bits)
print(f"Optimality-Cut: tight am Anker — Cut(z̄) = {vals[idx_z]:.2f} == Q(z̄) = {res.q_value:.2f}")

costs2 = costs1 + vals  # H_C-Diagonale Runde 2: direkte Kosten + max über Cuts (hier: 1 Cut)
probs_r2 = gm_qaoa(costs2, p=6)

print("verschiedene Kostenwerte:", np.unique(costs2).size,
      "| verschiedene P(z)-Werte:", np.unique(np.round(probs_r2, 14)).size)
print("P(argmin) =", round(float(probs_r2[np.argmin(costs2)]), 4),
      "| uniform =", round(1 / len(feas), 5))

fig = go.Figure()
fig.add_hline(y=1 / len(feas), line_dash="dash", line_color="#898781",
              annotation_text="uniform 1/|F|", annotation_font_color="#52514e")
fig.add_scatter(x=costs2, y=probs_r2, mode="markers",
                marker=dict(color="#2a78d6", size=5), showlegend=False)
fig.update_layout(
    title="Runde 2 (GM-QAOA p=6): kontinuierliche Kosten ⇒ kontinuierliches P(z)-Spektrum",
    xaxis_title="H_C-Diagonaleintrag von z ($, direkt + Cut)", yaxis_title="P(z)",
    width=750, height=420, plot_bgcolor="#fcfcfb",
)
fig.show()

## Der komplette Benders-Loop (Aufgaben 8 + 9)

Jetzt alles zusammen. Pro Runde:

1. **Master (Quantum):** $H_C$-Diagonale = direkte Kosten + punktweises Maximum aller
   bisherigen Optimality-Cuts → GM-QAOA → best-of-shots liefert $\bar z$.
2. **Subproblem (klassisch):** Gurobi löst das LP bei fixem $\bar z$.
   *Feasibel* → Optimality-Cut (Duals) verschärft $H_C$, und $g(\bar z) + Q(\bar z)$
   ist ein Kandidat für die obere Schranke (UB).
   *Infeasibel* → Feasibility-Cut (Farkas) streicht alle States mit demselben
   Unlösbarkeits-Beweis aus dem feasiblen Zustandsvektor — der Grover-Mixer über
   den Rest ist implizit (Gleichverteilung über das gefilterte Array).
3. **Schranken:** LB = exaktes Minimum des Master-Modells über die (verbliebene)
   Enumeration — im PoC gratis. Abbruch, sobald UB − LB ≤ gap_tol.

Die QAOA bleibt dabei ein *heuristischer Sampler* — die Benders-Struktur liefert
die Konvergenz, solange der Sampler das Master-Optimum je Runde findet
(bei 486 States mit 4096 Shots praktisch sicher).

In [24]:
from qc.benders import benders_loop, brute_force_optimum

loop = benders_loop(inst, max_rounds=30, gap_tol=1e-4, shots=4096, seed=0)
print(f"Terminierung: {loop.termination} nach {len(loop.rounds)} Runden, "
      f"Gap = {loop.gap:.6f}\n")
for r in loop.rounds:
    q = f"{r.q:8.2f}" if r.q is not None else "   —    "
    lb = f"{r.lb:9.2f}" if np.isfinite(r.lb) else "     -inf"
    cut = f"Feasibility-Cut: −{r.n_removed} States" if r.n_removed else "Optimality-Cut"
    print(f"Runde {r.round:2d}: z={r.z:0{inst.n_bits}b} {r.status:<10} Q={q}  "
          f"UB={r.ub:9.2f}  LB={lb}  |F|={r.n_states:3d}  {cut}")

Terminierung: gap nach 8 Runden, Gap = 0.000000

Runde  1: z=1010001001000000 infeasible Q=   —      UB=      inf  LB=     -inf  |F|=324  Feasibility-Cut: −162 States
Runde  2: z=1001000100100000 infeasible Q=   —      UB=      inf  LB=     -inf  |F|=162  Feasibility-Cut: −162 States
Runde  3: z=0010000000010101 infeasible Q=   —      UB=      inf  LB=     -inf  |F|= 81  Feasibility-Cut: −81 States
Runde  4: z=0001000000100000 infeasible Q=   —      UB=      inf  LB=     -inf  |F|= 54  Feasibility-Cut: −27 States
Runde  5: z=0100001000100100 infeasible Q=   —      UB=      inf  LB=     -inf  |F|= 27  Feasibility-Cut: −27 States
Runde  6: z=0010001000100010 optimal    Q=    0.00  UB=     0.00  LB=     -inf  |F|= 27  Optimality-Cut
Runde  7: z=0010001000101001 infeasible Q=   —      UB=     0.00  LB=   -12.50  |F|= 15  Feasibility-Cut: −12 States
Runde  8: z=0010000100101010 optimal    Q=   -0.24  UB=    -0.24  LB=   -12.50  |F|= 15  Optimality-Cut


In [25]:
rounds_x = [r.round for r in loop.rounds]
fig = go.Figure()
fig.add_scatter(x=rounds_x, y=[r.ub if np.isfinite(r.ub) else None for r in loop.rounds],
                mode="lines+markers", name="UB (bester gefundener Zielwert)",
                line=dict(color="#2a78d6"))
fig.add_scatter(x=rounds_x, y=[r.lb if np.isfinite(r.lb) else None for r in loop.rounds],
                mode="lines+markers", name="LB (Master-Schranke aus Cuts)",
                line=dict(color="#c9563c"))
fig.add_hline(y=loop.best_value, line_dash="dot", line_color="#898781",
              annotation_text=f"Optimum {loop.best_value:.2f} $",
              annotation_font_color="#52514e")
fig.update_layout(
    title="Benders-Konvergenz: obere und untere Schranke pro Runde",
    xaxis_title="Runde", yaxis_title="Gesamtkosten g(z) + q(z)  ($)",
    width=750, height=420, plot_bgcolor="#fcfcfb",
)
fig.show()

In [26]:
# Wie die Cuts das Sampling umlenken: P(z) über den Master-Kosten, Runde für Runde.
fig = go.Figure()
for k, r in enumerate(loop.rounds):
    fig.add_scatter(x=r.costs, y=r.probs, mode="markers",
                    marker=dict(color="#2a78d6", size=5),
                    name=f"Runde {r.round}", visible=(k == 0))
steps = [dict(method="update", label=f"Runde {r.round}",
              args=[{"visible": [j == k for j in range(len(loop.rounds))]},
                    {"title": f"Runde {r.round}: |F|={len(r.states)}, "
                              f"{'Feasibility-Cut' if r.n_removed else 'Optimality-Cut'}"}])
         for k, r in enumerate(loop.rounds)]
fig.update_layout(
    sliders=[dict(active=0, steps=steps)],
    title=f"Runde {loop.rounds[0].round}: |F|={len(loop.rounds[0].states)}",
    xaxis_title="Master-Kosten von z ($, direkt + max über Cuts)", yaxis_title="P(z)",
    width=750, height=460, plot_bgcolor="#fcfcfb",
)
fig.show()

In [22]:
# |F| pro Runde: Feasibility-Cuts räumen ganze State-Gruppen ab, Optimality-Cuts nicht.
fig = go.Figure()
fig.add_bar(x=rounds_x, y=[r.n_states for r in loop.rounds],
            marker_color=["#c9563c" if r.n_removed else "#2a78d6" for r in loop.rounds],
            text=[f"−{r.n_removed}" if r.n_removed else "" for r in loop.rounds],
            textposition="outside")
fig.update_layout(
    title="Feasible Set pro Runde — rot: Runde mit Feasibility-Cut (Anzahl entfernt)",
    xaxis_title="Runde", yaxis_title="|F| nach der Runde",
    width=750, height=380, plot_bgcolor="#fcfcfb",
)
fig.show()

In [23]:
from qc.instance import ROLES, decode

z_star, v_star, _ = brute_force_optimum(inst)   # exakte Referenz: 1 LP je feasiblem z
print(f"Loop:        z={loop.best_z:0{inst.n_bits}b}, Gesamtkosten {loop.best_value:.2f} $")
print(f"Brute force: z={z_star:0{inst.n_bits}b}, Gesamtkosten {v_star:.2f} $")
print(f"→ {'IDENTISCH' if abs(loop.best_value - v_star) <= 1e-3 else 'ABWEICHUNG!'}\n")

for t, slot in enumerate(decode(loop.best_z, inst)):
    on = "online" if inst.g_avail[t] else "OUTAGE"
    roles = ", ".join(role for role in ROLES if slot[role]) or "alles aus"
    print(f"Slot {t} ({on}): {roles}")
x = loop.best_x
print(f"\nx*: Import {np.round(x['p_imp'], 1)} kW | Export {np.round(x['p_exp'], 1)} kW | "
      f"Laden {np.round(x['p_ch'], 1)} kW | Entladen {np.round(x['p_dis'], 1)} kW")
print(f"    SoC-Verlauf {np.round(x['soc'], 1)} kWh | Peak {x['p_peak']:.1f} kW")

Loop:        z=0010000100101010, Gesamtkosten -0.24 $
Brute force: z=0010000000101010, Gesamtkosten -0.24 $
→ IDENTISCH

Slot 0 (online): dis, exp, b_mid
Slot 1 (OUTAGE): ch, b_mid

x*: Import [0. 0.] kW | Export [19.5  0. ] kW | Laden [0. 0.] kW | Entladen [250.   0.] kW
    SoC-Verlauf [434.1 434.1] kWh | Peak 0.0 kW


### Einordnung

Der Loop findet nachweislich (Gap = 0 gegen die exakte Master-Schranke, identisch mit
der Brute-Force-Referenz) das Optimum des hybriden Problems — diskrete Struktur aus
dem Quantum-Sampler, kontinuierliche Physik aus dem LP, verbunden durch Benders-Cuts.
Feasibility-Cuts räumen dabei ganze Gruppen strukturell erlaubter, aber physikalisch
unfortsetzbarer Konfigurationen ab (z. B. „served ohne Quelle"), Optimality-Cuts bauen
das untere Modell der kontinuierlichen Restkosten $q(z)$ auf, bis UB und LB zusammenfallen.

Grenzen bleiben wie in `QC_Ansatz.md` diskutiert: brute-force-Konstruktion von Mixer
und $H_C$, Konvergenzgarantie nur so stark wie der heuristische Master-Sampler.